# 02 — Makroekonomik Korelasyon & Regresyon Analizi

**Araştırma Sorusu:** Hangi makroekonomik göstergeler Apple'ın yıllık gelirini en güçlü şekilde açıklar?

Bu notebook'ta:
1. Korelasyon matrisi ve ısı haritası
2. Zaman serisi overlay grafikleri
3. OLS regresyon: `Apple Revenue ~ GDP + Fed Rate + CPI + USD`
4. Ekonomik yorum: hangi değişken neden önemli?

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

df = pd.read_csv('../data/processed/apple_macro_combined.csv', index_col='fiscal_year')
print(f'Dataset: {df.shape[0]} yıl, {df.shape[1]} değişken')
df.head()

## 1. Temel İstatistikler

In [ ]:
# Milyar dolara çevir — okuması kolay
df_display = df.copy()
for col in ['Total Revenue', 'Free Cash Flow', 'Net Income', 'Gross Profit', 'Operating Cash Flow']:
    if col in df_display.columns:
        df_display[col] = df_display[col] / 1e9

print('Finansal değişkenler: milyar USD')
df_display.describe().round(2)

## 2. Korelasyon Matrisi

In [ ]:
analysis_cols = ['Total Revenue', 'Free Cash Flow', 'fed_rate', 'cpi', 'gdp_growth', 'usd_index']
available_cols = [c for c in analysis_cols if c in df.columns]

df_analysis = df[available_cols].copy()
if 'Total Revenue' in df_analysis.columns:
    df_analysis['Total Revenue'] = df_analysis['Total Revenue'] / 1e9
if 'Free Cash Flow' in df_analysis.columns:
    df_analysis['Free Cash Flow'] = df_analysis['Free Cash Flow'] / 1e9

rename_map = {
    'Total Revenue': 'Revenue ($B)',
    'Free Cash Flow': 'FCF ($B)',
    'fed_rate': 'Fed Rate',
    'cpi': 'CPI',
    'gdp_growth': 'GDP Growth',
    'usd_index': 'USD Index',
}
df_analysis = df_analysis.rename(columns=rename_map)

corr = df_analysis.corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, annot=True, fmt='.2f', cmap='RdYlGn',
    center=0, vmin=-1, vmax=1,
    mask=mask, ax=ax, linewidths=0.5,
    annot_kws={'size': 10}
)
ax.set_title('Korelasyon Matrisi: Apple Finansallar × Makro Göstergeler', fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('../data/processed/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

if 'Revenue ($B)' in corr.columns:
    print('\nApple Geliri ile Korelasyonlar:')
    print(corr['Revenue ($B)'].drop('Revenue ($B)').sort_values(key=abs, ascending=False).to_string())

## 3. Zaman Serisi Overlay: Gelir vs Makro

In [ ]:
macro_vars = [
    ('fed_rate', 'Fed Funds Rate (%)', '#e74c3c'),
    ('gdp_growth', 'GDP Büyümesi (%)', '#27ae60'),
    ('cpi', 'CPI Endeksi', '#f39c12'),
    ('usd_index', 'USD Endeksi', '#8e44ad'),
]

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for i, (col, label, color) in enumerate(macro_vars):
    if col not in df.columns or 'Total Revenue' not in df.columns:
        continue
    ax1 = axes[i]
    ax2 = ax1.twinx()

    ax1.bar(df.index, df['Total Revenue'] / 1e9, alpha=0.35, color='#3498db', label='Revenue ($B)')
    ax2.plot(df.index, df[col], color=color, linewidth=2.5, marker='o', markersize=5, label=label)

    ax1.set_ylabel('Apple Gelir ($B)', color='#3498db')
    ax2.set_ylabel(label, color=color)
    ax1.set_title(f'Apple Gelir vs {label}', fontweight='bold')
    ax1.grid(axis='y', alpha=0.2)

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=9)

plt.suptitle('Apple Yıllık Geliri vs Makroekonomik Göstergeler', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../data/processed/revenue_vs_macro_overlay.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. OLS Regresyon Analizi

In [ ]:
reg_cols = ['gdp_growth', 'fed_rate', 'cpi', 'usd_index']
available_reg = [c for c in reg_cols if c in df.columns]

reg_df = df[['Total Revenue'] + available_reg].dropna()

y = reg_df['Total Revenue'] / 1e9
X = reg_df[available_reg]
X = sm.add_constant(X)

model = sm.OLS(y, X).fit()
print(model.summary())

In [ ]:
# Katsayılar ve istatistiksel anlamlılık özeti
coef_df = pd.DataFrame({
    'Katsayı': model.params.drop('const'),
    'p-değeri': model.pvalues.drop('const'),
    'Anlamlı (p<0.05)': model.pvalues.drop('const') < 0.05
}).round(4)

print(f'R² = {model.rsquared:.4f} — Modelin Açıklama Gücü: {model.rsquared*100:.1f}%')
print(f'Adj. R² = {model.rsquared_adj:.4f}')
print()
print(coef_df.to_string())

print('\n--- Ekonomik Yorum ---')
for var, row in coef_df.iterrows():
    direction = 'artıyor' if row['Katsayı'] > 0 else 'azalıyor'
    sig = '(istatistiksel olarak anlamlı)' if row['Anlamlı (p<0.05)'] else '(anlamlı değil)'
    print(f"  {var}: 1 birim artışta Apple geliri {abs(row['Katsayı']):.2f}B$ {direction} {sig}")

In [ ]:
# Katsayı görselleştirmesi
coefs = model.params.drop('const')
errors = model.conf_int().drop('const')
errors.columns = ['low', 'high']

colors = ['#2ecc71' if c > 0 else '#e74c3c' for c in coefs]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(coefs.index, coefs.values, color=colors, alpha=0.8, height=0.5)
ax.errorbar(
    coefs.values, range(len(coefs)),
    xerr=[coefs.values - errors['low'].values, errors['high'].values - coefs.values],
    fmt='none', color='black', capsize=5, linewidth=1.5
)
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('Katsayı (Apple Geliri üzerindeki etki, $B)')
ax.set_title('OLS Regresyon Katsayıları\n(Apple Revenue ~ Makro Göstergeler)', fontweight='bold')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('../data/processed/regression_coefficients.png', dpi=150, bbox_inches='tight')
plt.show()

## Özet Bulgular

| Gösterge | Apple'a Etkisi | Mekanizma |
|----------|---------------|----------|
| **GDP Büyümesi** | En güçlü pozitif | Tüketici harcamaları → iPhone/Mac satışları |
| **Fed Faizi** | Negatif | Discount rate artışı → değerleme baskısı + tüketici borçlanma maliyeti |
| **CPI (Enflasyon)** | Karışık | Maliyet baskısı vs. fiyatlama gücü |
| **USD Endeksi** | Negatif | Apple gelirinin ~%60'ı uluslararası; güçlü dolar çeviri kaybı yaratır |

Sonraki adım: **03_dcf_model.ipynb** — bu regresyon bulguları FCF büyüme oranı varsayımlarını kalibre edecek.